In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import tensorflow as tf
from collections import deque
import random


/home/shreyash/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shreyash/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
2025-07-02 10:09:06.273492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751465346.284888  319039 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751465346.288295  319039 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugi

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"



In [ ]:
# Load the .npz file
loaded_data = np.load("cnn_results_combined.npz")
true_labels = loaded_data['true_labels']
softmax_outputs = loaded_data['softmax_outputs']
predicted_labels = loaded_data['predicted_labels']

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# === Setup ===
total_size = len(true_labels)
val_size = int(0.2 * total_size)  # 20% validation
all_indices = np.arange(total_size)

# Random shuffle for unbiased split
np.random.seed(42)  # for reproducibility
np.random.shuffle(all_indices)

# Split indices
val_indices = all_indices[:val_size]
remaining_indices = all_indices[val_size:]  # 80% test

# === Sanity Check ===
print("=== Set Sizes ===")
print(f"Validation Set: {len(val_indices)}")
print(f"Test Set:       {len(remaining_indices)}")

# Optional: check distribution of labels in each set
val_labels = true_labels[val_indices]
test_labels = true_labels[remaining_indices]

val_known = np.sum(val_labels != 10)
val_unknown = np.sum(val_labels == 10)

test_known = np.sum(test_labels != 10)
test_unknown = np.sum(test_labels == 10)

print("\n=== Label Distribution (for sanity, not used in split) ===")
print(f"Validation Known:   {val_known}")
print(f"Validation Unknown: {val_unknown}")
print(f"Test Known:         {test_known}")
print(f"Test Unknown:       {test_unknown}")


=== Set Sizes ===
Validation Set: 2503
Test Set:       10016

=== Label Distribution (for sanity, not used in split) ===
Validation Known:   2058
Validation Unknown: 445
Test Known:         8234
Test Unknown:       1782


In [ ]:
def compute_entropy(softmax_outputs):

    epsilon = 1e-12  # to avoid log(0)
    softmax_clipped = np.clip(softmax_outputs, epsilon, 1. - epsilon)
    entropy = -np.sum(softmax_clipped * np.log(softmax_clipped), axis=1)
    return entropy

# === Compute Entropy ===
val_softmax_outputs = softmax_outputs[val_indices]
test_softmax_outputs = softmax_outputs[remaining_indices]

val_entropy = compute_entropy(val_softmax_outputs)
test_entropy = compute_entropy(test_softmax_outputs)

# === Stats Check ===
print(f"Validation entropy stats - min: {val_entropy.min():.4f}, max: {val_entropy.max():.4f}, mean: {val_entropy.mean():.4f}")
print(f"Test entropy stats       - min: {test_entropy.min():.4f}, max: {test_entropy.max():.4f}, mean: {test_entropy.mean():.4f}")


Validation entropy stats - min: 0.0001, max: 1.6644, mean: 0.2085
Test entropy stats       - min: 0.0001, max: 1.7733, mean: 0.2137


In [ ]:
from scipy.stats import entropy

def compute_entropy(softmax_probs):

    epsilon = 1e-12
    softmax_clipped = np.clip(softmax_probs, epsilon, 1. - epsilon)
    return entropy(softmax_clipped.T)  # entropy across classes for each sample

# === Extract softmax outputs for validation set ===
val_softmax_outputs = softmax_outputs[val_indices]  # shape: (n_val_samples, n_classes)

# === Compute p1 (max confidence), p1 - p2, and entropy ===
val_p1 = np.max(val_softmax_outputs, axis=1)
sorted_preds = np.sort(val_softmax_outputs, axis=1)
val_p1_p2_diff = sorted_preds[:, -1] - sorted_preds[:, -2]
val_entropy = compute_entropy(val_softmax_outputs)

# === Define top/bottom 5% size ===
n_val = len(val_indices)
n_top_bottom = int(0.05 * n_val)

# === Get top 5% high-confidence based on p1 only ===
sorted_indices = np.argsort(val_p1)
low_conf_top_indices = sorted_indices[:n_top_bottom]        # bottom 5% → low confidence
high_conf_top_indices = sorted_indices[-n_top_bottom:]      # top 5% → high confidence

# === Create boolean masks (for excluding in training) ===
high_conf_mask = np.zeros(n_val, dtype=bool)
low_conf_mask = np.zeros(n_val, dtype=bool)
high_conf_mask[high_conf_top_indices] = True
low_conf_mask[low_conf_top_indices] = True

# === Optional Debug Output ===
print(f"High-confidence samples selected: {len(high_conf_top_indices)}")
print(f"Low-confidence samples selected:  {len(low_conf_top_indices)}")
print(f"Mean p1: {np.mean(val_p1):.4f}")
print(f"Mean p1-p2 diff: {np.mean(val_p1_p2_diff):.4f}")
print(f"Mean entropy: {np.mean(val_entropy):.4f}")


High-confidence samples selected: 125
Low-confidence samples selected:  125
Mean p1: 0.9315
Mean p1-p2 diff: 0.8949
Mean entropy: 0.2085


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class DQNAgent:
    def __init__(self, state_size=3, action_size=2):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.990
        self.learning_rate = 0.001
        self.model = self._build_model()

    def _build_model(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.state_size,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate), loss='mse')
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randint(0, self.action_size - 1)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * np.amax(self.model.predict(next_state, verbose=0)[0])
            q_values = self.model.predict(state, verbose=0)
            q_values[0][action] = target
            self.model.fit(state, q_values, epochs=1, verbose=0)
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

# === Initialization ===
agent = DQNAgent(state_size=3, action_size=2)
batch_size = 32
episodes = 30

val_pred_confidence = np.zeros(n_val, dtype=int)

# Initial centroids from top/bottom 5% p1 confidence
high_conf_list = list(high_conf_top_indices)
low_conf_list = list(low_conf_top_indices)

centroid_high = np.array([
    np.mean(val_p1[high_conf_list]),
    np.mean(val_p1_p2_diff[high_conf_list]),
    np.mean(val_entropy[high_conf_list])
])
centroid_low = np.array([
    np.mean(val_p1[low_conf_list]),
    np.mean(val_p1_p2_diff[low_conf_list]),
    np.mean(val_entropy[low_conf_list])
])
high_count = len(high_conf_list)
low_count = len(low_conf_list)

# Remaining samples (excluding anchors)
rest_indices = np.where(~high_conf_mask & ~low_conf_mask)[0]
np.random.shuffle(rest_indices)
subsample_size = min(1500, len(rest_indices))
rest_indices = rest_indices[:subsample_size]

# === Training Loop ===
for episode in range(episodes):
    total_reward = 0
    n_processed = 0
    action_stats = {0: 0, 1: 0}

    for i, idx in enumerate(rest_indices):
        state = np.array([val_p1[idx], val_p1_p2_diff[idx], val_entropy[idx]]).reshape(1, -1)
        action = agent.act(state)
        action_stats[action] += 1

        # Compute similarity to both centroids
        sim_high = cosine_similarity(state, centroid_high.reshape(1, -1))[0][0]
        sim_low = cosine_similarity(state, centroid_low.reshape(1, -1))[0][0]

        # Reward = similarity to closest match, penalty otherwise
        reward_magnitude = max(sim_high, sim_low)
        best_action = 1 if sim_high > sim_low else 0
        reward = reward_magnitude if action == best_action else -reward_magnitude
        reward = np.clip(reward, -1.0, 1.0)

        total_reward += reward
        n_processed += 1
        val_pred_confidence[idx] = action

        # Only update centroid if similarity is strong (prevents noise)
        similarity_threshold = 0.75
        if action == 1 and sim_high > similarity_threshold:
            centroid_high = (centroid_high * high_count + state.flatten()) / (high_count + 1)
            high_count += 1
        elif action == 0 and sim_low > similarity_threshold:
            centroid_low = (centroid_low * low_count + state.flatten()) / (low_count + 1)
            low_count += 1

        # Next state for replay buffer
        next_idx = random.choice(rest_indices)
        next_state = np.array([val_p1[next_idx], val_p1_p2_diff[next_idx], val_entropy[next_idx]]).reshape(1, -1)
        done = (i == len(rest_indices) - 1)
        agent.remember(state, action, reward, next_state, done)

    # Train from memory
    agent.replay(batch_size)

    # === Logging ===
    avg_reward = total_reward / n_processed
    print(f"\n--- Episode {episode + 1}/{episodes} ---")
    print(f"Avg Reward: {avg_reward:.4f} | ε: {agent.epsilon:.4f}")
    print(f"High Conf Count: {high_count} | Low Conf Count: {low_count}")
    print(f"Actions → High: {action_stats[1]}, Low: {action_stats[0]}")
    print(f"Centroid High: {centroid_high}")


2025-07-02 10:10:07.423620: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-07-02 10:10:07.423641: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-07-02 10:10:07.423646: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-07-02 10:10:07.423649: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-07-02 10:10:07.423653: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: gokhan-Lambda-Vector
2025-07-02 10:10:07.423655: I external/local_xla/xla/stream_execut


--- Episode 1/30 ---
Avg Reward: -0.0359 | ε: 0.9900
High Conf Count: 762 | Low Conf Count: 222
Actions → High: 734, Low: 766
Centroid High: [0.9953695  0.9917472  0.02180215]

--- Episode 2/30 ---
Avg Reward: -0.0492 | ε: 0.9801
High Conf Count: 1385 | Low Conf Count: 328
Actions → High: 709, Low: 791
Centroid High: [0.99429476 0.9898776  0.02606924]

--- Episode 3/30 ---
Avg Reward: 0.0547 | ε: 0.9703
High Conf Count: 2102 | Low Conf Count: 416
Actions → High: 816, Low: 684
Centroid High: [0.99363804 0.98864686 0.02831975]

--- Episode 4/30 ---
Avg Reward: -0.0135 | ε: 0.9606
High Conf Count: 2756 | Low Conf Count: 518
Actions → High: 749, Low: 751
Centroid High: [0.99351555 0.9883608  0.02879887]

--- Episode 5/30 ---
Avg Reward: 0.0372 | ε: 0.9510
High Conf Count: 3450 | Low Conf Count: 624
Actions → High: 783, Low: 717
Centroid High: [0.9931992  0.9877751  0.02982462]

--- Episode 6/30 ---
Avg Reward: 0.0347 | ε: 0.9415
High Conf Count: 4147 | Low Conf Count: 726
Actions → High: 

In [ ]:
from sklearn.metrics import accuracy_score, f1_score



# Extract test features consistent with training:
test_p1 = np.max(test_softmax_outputs, axis=1)
sorted_test_preds = np.sort(test_softmax_outputs, axis=1)
test_p1_p2_diff = sorted_test_preds[:, -1] - sorted_test_preds[:, -2]
test_entropy = compute_entropy(test_softmax_outputs)

test_features = np.stack([test_p1, test_p1_p2_diff, test_entropy], axis=1)

# Predict confidence (action 0 or 1) using trained DQN model
dqn_predictions = agent.model.predict(test_features, verbose=0)
predicted_actions = np.argmax(dqn_predictions, axis=1)  # 0 or 1

# Prepare true binary confidence labels:
known_labels = np.arange(10)  # your known class indices
test_true_labels = true_labels[remaining_indices]
true_confidence = np.isin(test_true_labels, known_labels).astype(int)

# Evaluate performance:
accuracy = accuracy_score(true_confidence, predicted_actions)
f1 = f1_score(true_confidence, predicted_actions)

# Count correct predictions in known and unknown separately:
correct_known = np.sum((predicted_actions == 1) & (true_confidence == 1))
total_known = np.sum(true_confidence == 1)

correct_unknown = np.sum((predicted_actions == 0) & (true_confidence == 0))
total_unknown = np.sum(true_confidence == 0)

print("=== Test Set Confidence Prediction ===")
print(f"Total samples: {len(test_true_labels)}")
print(f"Known samples: {total_known}")
print(f"Unknown samples: {total_unknown}")
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Overall F1 Score: {f1:.4f}")
print(f"Correct Known predictions: {correct_known} / {total_known}")
print(f"Correct Unknown predictions: {correct_unknown} / {total_unknown}")


=== Test Set Confidence Prediction ===
Total samples: 10016
Known samples: 8234
Unknown samples: 1782
Overall Accuracy: 0.9577
Overall F1 Score: 0.9744
Correct Known predictions: 8075 / 8234
Correct Unknown predictions: 1517 / 1782


#Evaluation of CNN-DQN

In [ ]:
print("\n=== CNN Predictions Breakdown for Known Classes ===")
for cls in known_labels:
    cls_mask = (true_known_labels == cls)
    pred_counts = Counter(cnn_pred_known_labels[cls_mask])
    print(f"True class {cls}: CNN predicted counts: {dict(pred_counts)}")



=== CNN Predictions Breakdown for Known Classes ===
True class 0: CNN predicted counts: {0: 915, 8: 26, 1: 17, 9: 53, 3: 31, 6: 6, 5: 10, 4: 1, 2: 12, 7: 29}
True class 1: CNN predicted counts: {1: 1099, 0: 1}
True class 2: CNN predicted counts: {2: 1082, 3: 8, 8: 9, 5: 1}
True class 3: CNN predicted counts: {3: 1096, 2: 1, 1: 2, 5: 1}
True class 4: CNN predicted counts: {4: 1079, 2: 7, 5: 12, 1: 1, 0: 1}
True class 5: CNN predicted counts: {5: 1085, 4: 11, 0: 1, 3: 1, 2: 2}
True class 6: CNN predicted counts: {6: 1088, 5: 9, 3: 2, 8: 1}
True class 7: CNN predicted counts: {7: 1093, 4: 2, 0: 1, 1: 1, 3: 1, 5: 1}
True class 8: CNN predicted counts: {8: 1086, 0: 8, 7: 3, 6: 2, 3: 1}
True class 9: CNN predicted counts: {9: 392, 0: 1}


In [ ]:
from collections import defaultdict

# Load results
loaded = np.load("cnn_results_combined.npz")
true_labels = loaded['true_labels']
predicted_labels = loaded['predicted_labels']
softmax_outputs = loaded['softmax_outputs']

# Known and unknown labels
known_labels = np.arange(10)
unknown_label = 10

# === Use test set only ===
test_indices = remaining_indices  # from your earlier val/test split
test_true = true_labels[test_indices]
test_pred = predicted_labels[test_indices]
test_softmax = softmax_outputs[test_indices]

# Extract test features for DQN
p1 = np.max(test_softmax, axis=1)
sorted_preds = np.sort(test_softmax, axis=1)
p1_p2_diff = sorted_preds[:, -1] - sorted_preds[:, -2]
entropy = -np.sum(test_softmax * np.log(test_softmax + 1e-10), axis=1)
test_features = np.stack([p1, p1_p2_diff, entropy], axis=1)

# Get DQN predictions
dqn_preds = np.argmax(agent.model.predict(test_features, verbose=0), axis=1)  # 0 = unknown, 1 = known

# === Only analyze known-class test samples ===
mask_known = test_true != unknown_label
true_known = test_true[mask_known]
pred_known = test_pred[mask_known]
dqn_known = dqn_preds[mask_known]

# === Count per-class accuracy of CNN+DQN ===
summary = defaultdict(lambda: {'correct': 0, 'total': 0})

for i in range(len(true_known)):
    t = true_known[i]
    p = pred_known[i]
    dqn = dqn_known[i]

    summary[t]['total'] += 1
    if t == p and dqn == 1:
        summary[t]['correct'] += 1

# === Print Summary ===
print("\n=== CNN + DQN Accuracy on Known Test Samples ===")
for cls in sorted(summary):
    correct = summary[cls]['correct']
    total = summary[cls]['total']
    acc = correct / total if total > 0 else 0
    print(f"Class {cls}: {correct} / {total} ({acc:.2%})")



=== CNN + DQN Accuracy on Known Test Samples ===
Class 0: 672 / 867 (77.51%)
Class 1: 880 / 882 (99.77%)
Class 2: 864 / 880 (98.18%)
Class 3: 882 / 898 (98.22%)
Class 4: 846 / 868 (97.47%)
Class 5: 870 / 884 (98.42%)
Class 6: 866 / 876 (98.86%)
Class 7: 872 / 880 (99.09%)
Class 8: 859 / 871 (98.62%)
Class 9: 326 / 328 (99.39%)
